In [2]:
import sqlite3, pandas as pd

con = sqlite3.connect('cohort.db')

def q(sql): return pd.read_sql(sql, con)

q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,patients
1,samples
2,genes
3,mutations


In [3]:
q("PRAGMA table_info(patients)")

,cid,name,type,notnull,dflt_value,pk
0,0,patient_id,TEXT,0,None,1
1,1,sex,TEXT,0,None,0
2,2,age_at_diagnosis,REAL,0,None,0
3,3,stage,TEXT,0,None,0
4,4,vital_status,TEXT,0,None,0
5,5,followup_months,REAL,0,None,0


In [4]:
q("PRAGMA table_info(samples)")

,cid,name,type,notnull,dflt_value,pk
0,0,sample_id,TEXT,0,None,1
1,1,patient_id,TEXT,1,None,0
2,2,tissue_source_site,TEXT,0,None,0
3,3,tmb_nonsynonymous,REAL,0,None,0
4,4,aneuploidy_score,REAL,0,None,0
5,5,msi_mantis_score,REAL,0,None,0


In [5]:
q("PRAGMA table_info(mutations)")

,cid,name,type,notnull,dflt_value,pk
0,0,mutation_id,INTEGER,0,None,1
1,1,sample_id,TEXT,1,None,0
2,2,gene_symbol,TEXT,1,None,0
3,3,variant_classification,TEXT,0,None,0
4,4,variant_type,TEXT,0,None,0
5,5,protein_change,TEXT,0,None,0
6,6,chromosome,TEXT,0,None,0
7,7,start_pos,INTEGER,0,None,0
8,8,end_pos,INTEGER,0,None,0
9,9,t_ref_count,INTEGER,0,None,0


In [6]:
q("PRAGMA table_info(genes)")

,cid,name,type,notnull,dflt_value,pk
0,0,gene_symbol,TEXT,0,None,1
1,1,entrez_gene_id,INTEGER,0,None,0
2,2,chromosome,TEXT,0,None,0


In [7]:
q("PRAGMA foreign_key_list(samples)")

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,patients,patient_id,patient_id,NO ACTION,NO ACTION,NONE


In [8]:
q("PRAGMA foreign_key_list(mutations)")

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,genes,gene_symbol,gene_symbol,NO ACTION,NO ACTION,NONE
1,1,0,samples,sample_id,sample_id,NO ACTION,NO ACTION,NONE


In [9]:
q("SELECT COUNT(*) AS n_patients FROM patients")

,n_patients
0,566


In [10]:

q("PRAGMA table_info(samples)")

,cid,name,type,notnull,dflt_value,pk
0,0,sample_id,TEXT,0,None,1
1,1,patient_id,TEXT,1,None,0
2,2,tissue_source_site,TEXT,0,None,0
3,3,tmb_nonsynonymous,REAL,0,None,0
4,4,aneuploidy_score,REAL,0,None,0
5,5,msi_mantis_score,REAL,0,None,0


In [11]:
q("PRAGMA foreign_key_list(samples)")

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,patients,patient_id,patient_id,NO ACTION,NO ACTION,NONE


In [12]:
q("PRAGMA table_info(mutations)")

,cid,name,type,notnull,dflt_value,pk
0,0,mutation_id,INTEGER,0,None,1
1,1,sample_id,TEXT,1,None,0
2,2,gene_symbol,TEXT,1,None,0
3,3,variant_classification,TEXT,0,None,0
4,4,variant_type,TEXT,0,None,0
5,5,protein_change,TEXT,0,None,0
6,6,chromosome,TEXT,0,None,0
7,7,start_pos,INTEGER,0,None,0
8,8,end_pos,INTEGER,0,None,0
9,9,t_ref_count,INTEGER,0,None,0


In [13]:
q("PRAGMA foreign_key_list(mutations)")

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,genes,gene_symbol,gene_symbol,NO ACTION,NO ACTION,NONE
1,1,0,samples,sample_id,sample_id,NO ACTION,NO ACTION,NONE


In [14]:
q("PRAGMA table_info(genes)")

,cid,name,type,notnull,dflt_value,pk
0,0,gene_symbol,TEXT,0,None,1
1,1,entrez_gene_id,INTEGER,0,None,0
2,2,chromosome,TEXT,0,None,0


In [15]:
q("PRAGMA foreign_key_list(genes)")

,id,seq,table,from,to,on_update,on_delete,match


In [16]:
###### TASK 2



q("SELECT COUNT(*) AS n_patients FROM patients")

,n_patients
0,566


In [17]:
tp53 = q('''
SELECT p.patient_id, COUNT(m.mutation_id) AS tp53_count
FROM patients p
LEFT JOIN samples s ON s.patient_id = p.patient_id
LEFT JOIN mutations m ON m.sample_id = s.sample_id
    AND UPPER(TRIM(m.gene_symbol)) = 'TP53'
    AND m.variant_classification != 'Silent'
GROUP BY p.patient_id
ORDER BY p.patient_id
''')

print(len(tp53), 'rows')
tp53.head()

566 rows


,patient_id,tp53_count
0,TCGA-05-4244,0
1,TCGA-05-4249,0
2,TCGA-05-4250,0
3,TCGA-05-4382,1
4,TCGA-05-4384,1


In [18]:
gene_counts = q('''
WITH target_genes(gene_symbol) AS (
    VALUES ('TP53'), ('KRAS'), ('EGFR'),
           ('STK11'), ('KEAP1')
)
SELECT p.patient_id,
       g.gene_symbol,
       COUNT(m.mutation_id) AS mutation_count
FROM patients p
CROSS JOIN target_genes g
LEFT JOIN samples s ON s.patient_id = p.patient_id
LEFT JOIN mutations m ON m.sample_id = s.sample_id
    AND UPPER(TRIM(m.gene_symbol)) = g.gene_symbol
    AND m.variant_classification != 'Silent'
GROUP BY p.patient_id, g.gene_symbol
ORDER BY p.patient_id, g.gene_symbol
''')

print(len(gene_counts), 'rows')
gene_counts.head(10)

2830 rows


,patient_id,gene_symbol,mutation_count
0,TCGA-05-4244,EGFR,0
1,TCGA-05-4244,KEAP1,0
2,TCGA-05-4244,KRAS,1
3,TCGA-05-4244,STK11,0
4,TCGA-05-4244,TP53,0
5,TCGA-05-4249,EGFR,0
6,TCGA-05-4249,KEAP1,0
7,TCGA-05-4249,KRAS,1
8,TCGA-05-4249,STK11,0
9,TCGA-05-4249,TP53,0


In [19]:
site_stage = q('''
SELECT s.tissue_source_site,
       p.stage,
       COUNT(DISTINCT p.patient_id) AS patient_count
FROM patients p
LEFT JOIN samples s ON s.patient_id = p.patient_id
GROUP BY s.tissue_source_site, p.stage
ORDER BY s.tissue_source_site, p.stage
''')

print(len(site_stage), 'rows')
site_stage

142 rows


,tissue_source_site,stage,patient_count
0,ABS - IUPUI,IA,5
1,ABS - IUPUI,IB,5
2,ABS - IUPUI,IIA,3
3,ABS - IUPUI,IIB,1
4,ABS - IUPUI,IIIA,2
...,...,...,...
137,Washington University - Rush University,IIA,1
138,Washington University - Rush University,IIIA,2
139,Washington University - St. Louis,IA,4
140,Washington University - St. Louis,IB,1


In [20]:
q('''
SELECT COUNT(DISTINCT p.patient_id) AS total_patients
FROM patients p
LEFT JOIN samples s ON s.patient_id = p.patient_id
''')

,total_patients
0,566


In [21]:
print(site_stage['patient_count'].sum())

566


In [22]:
clinical = q('''
SELECT p.patient_id,
       p.sex,
       p.age_at_diagnosis,
       p.stage,
       s.tissue_source_site,
       s.tmb_nonsynonymous
FROM patients p
LEFT JOIN samples s ON s.patient_id = p.patient_id
ORDER BY p.patient_id
''')

print(len(clinical), 'rows')
print(clinical['patient_id'].nunique(), 'unique patients')

clinical.head()

566 rows
566 unique patients


,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667


In [23]:
gene_wide = gene_counts.pivot(
    index='patient_id',
    columns='gene_symbol',
    values='mutation_count'
).reset_index()

gene_wide.columns.name = None

print(len(gene_wide), 'rows')
gene_wide.head()

566 rows


,patient_id,EGFR,KEAP1,KRAS,STK11,TP53
0,TCGA-05-4244,0,0,1,0,0
1,TCGA-05-4249,0,0,1,0,0
2,TCGA-05-4250,0,0,1,0,0
3,TCGA-05-4382,2,0,0,0,1
4,TCGA-05-4384,0,1,0,0,1


In [24]:
feature_table = clinical.merge(
    gene_wide,
    on='patient_id',
    how='left',
    validate='one_to_one'
)

print(len(feature_table), 'rows')
print(feature_table['patient_id'].nunique(), 'unique patients')

feature_table.head()

566 rows
566 unique patients


,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous,EGFR,KEAP1,KRAS,STK11,TP53
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000,0,0,1,0,0
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000,0,0,1,0,0
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000,0,0,1,0,0
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333,2,0,0,0,1
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667,0,1,0,0,1


In [25]:
genes = ['TP53', 'KRAS', 'EGFR', 'STK11', 'KEAP1']

feature_table[genes] = feature_table[genes].fillna(0).astype(int)

feature_table = feature_table[
    ['patient_id', 'sex', 'age_at_diagnosis', 'stage',
     'tissue_source_site', 'tmb_nonsynonymous'] + genes
]

print(feature_table.shape)
feature_table.head()

(566, 11)


,patient_id,sex,age_at_diagnosis,stage,tissue_source_site,tmb_nonsynonymous,TP53,KRAS,EGFR,STK11,KEAP1
0,TCGA-05-4244,male,70.0,IV,Indivumed,6.400000,0,1,0,0,0
1,TCGA-05-4249,male,67.0,IB,Indivumed,10.000000,0,1,0,0,0
2,TCGA-05-4250,female,79.0,IIIA,Indivumed,10.500000,0,1,0,0,0
3,TCGA-05-4382,male,68.0,IB,Indivumed,51.733333,1,0,2,0,0
4,TCGA-05-4384,male,66.0,IIIA,Indivumed,3.966667,1,0,0,0,1


In [26]:
feature_table.isna().sum()

patient_id             0
sex                   52
age_at_diagnosis      71
stage                 54
tissue_source_site     0
tmb_nonsynonymous      0
TP53                   0
KRAS                   0
EGFR                   0
STK11                  0
KEAP1                  0
dtype: int64

In [27]:
print("Patients before joins:", len(q("SELECT patient_id FROM patients")))
print("Patients after joins:", len(feature_table))
print("Unique patients:", feature_table['patient_id'].nunique())

Patients before joins: 566
Patients after joins: 566
Unique patients: 566


In [28]:
genes = ['TP53', 'KRAS', 'EGFR', 'STK11', 'KEAP1']

print("Missing gene counts:")
print(feature_table[genes].isna().sum())

print("Patients with zero mutations in all five genes:")
print((feature_table[genes].sum(axis=1) == 0).sum())

Missing gene counts:
TP53     0
KRAS     0
EGFR     0
STK11    0
KEAP1    0
dtype: int64
Patients with zero mutations in all five genes:
109


In [29]:
feature_table.to_csv(
    "feature_table.tsv",
    sep="\t",
    index=False,
    na_rep="NA"
)

print("Feature table saved!")

Feature table saved!


In [30]:
feature_table.isna().sum()

patient_id             0
sex                   52
age_at_diagnosis      71
stage                 54
tissue_source_site     0
tmb_nonsynonymous      0
TP53                   0
KRAS                   0
EGFR                   0
STK11                  0
KEAP1                  0
dtype: int64